# Quantify invasion

**Purpose.** Classify pathogens as intracellular or extracellular using differential-staining measurements and summarize invasion efficiency.

**Recommended use.** Use for invasion or attachment assays with an extracellular marker.

**Primary outputs.** Per-object invasion labels and per-well or per-condition invasion estimates.

---

> Paths in this notebook are placeholders. Set `src` and any other required path to the experimental data before execution.
> spaCR writes outputs within, or immediately adjacent to, the configured source directory unless an explicit output path is set.

## 1. Verify the environment

The following cell reports the installed spaCR version. Import errors must be resolved before the analysis cells are executed. GPU acceleration is optional and depends on the selected workflow and installed computational backend.

In [ ]:
import spacr
from spacr.version import version_str

print(version_str)

## 2. API entry point

This workflow calls the following public function:

- [`spacr.submodules.analyze_invasion`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_invasion)

```python
analyze_invasion(settings)
```

Parameter definitions and defaults are generated from the same public API and are listed in the settings reference below.

In [ ]:
from spacr.submodules import analyze_invasion

## 3. Settings and API reference

Review the parameter definitions here, then edit the values in the categorized code cells below. Required settings must be supplied. Optional settings retain the displayed default when unchanged. Conditionally required settings are necessary only for the indicated analysis branch. Defaults and descriptions are generated from the installed spaCR version so that the notebook remains aligned with the public API.

### [`spacr.submodules.analyze_invasion`](https://einarolafsson.github.io/spacr/api/spacr/submodules/index.html#spacr.submodules.analyze_invasion)


#### Assay Inputs

- **`src`** *(required)* — (str, path) - Folder the current step reads from and writes into: raw images for mask generation, the merged/ folder of .npy stacks for measure, the plate root for dataset/regression steps, or the folder of .fastq.gz reads for sequencing. Outputs (stack/, masks/, measurements/measurements.db, datasets/, results/) are created inside it. A list of paths, or a "['a','b']" string, processes several plates in one run. No usable default: the settings factories fill a placeholder ('path' or '/path/to/src'), so this must be supplied.
- **`parasite_table`** *(optional)* — (str) - Table in measurements/measurements.db holding one row per segmented parasite. It is read directly rather than through the usual merge, because that merge collapses pathogen rows onto their host cell and would sum several parasites' stain intensities into a single row. Change it only if measure_crop wrote the parasite objects under a non-standard name. Default 'pathogen'.
- **`compartment`** *(optional)* — (str) - Prefix the per-object measurement columns carry, so 'pathogen' selects pathogen_area and pathogen_channel_1_percentile_95. It must match the object type the table actually holds, or the area filters and the intensity statistic resolve to columns that do not exist and the run aborts naming them. Default 'pathogen'.

#### Channels & Intensity

- **`outside_channel`** *(required)* — (int) - Zero-indexed channel of the pre-permeabilisation antibody, which reaches only parasites still outside the host cell. Every classification the assay makes is a threshold on this channel, so pointing it at the wrong stain silently converts the readout into whatever that channel measures. Note this is a channel index, not measure.py's &lt;object&gt;_channel_&lt;n&gt;_outside_* columns, which are the ring just outside an object's own mask. Default 1.
- **`total_channel`** *(required)* — (int or None) - Zero-indexed channel of the post-permeabilisation antibody that stains every parasite. Nothing is classified from it; it only supplies the intensity that min_total_intensity filters on, so an incorrect value costs nothing until that filter is switched on. Default 0.
- **`intensity_statistic`** *(optional)* — (str) - Which per-object statistic of the pre-permeabilisation channel is thresholded. That stain sits on the parasite's SURFACE, so the object's mean divides a rim by the whole area and reads a large parasite as dimmer than a small one stained identically -- a bias that turns outside parasites into apparent inside ones as size varies. A percentile of the rim pixels is the honest choice. Default 'mean'.
- **`background_correction`** *(optional)* — (str) - Per-object local background subtracted from the outside-stain statistic before thresholding. 'auto' uses the median of the five-pixel ring outside the parasite mask, which removes a per-field offset without a flat-field image; 'none' subtracts nothing. Switching it on can backfire: a brightly stained attached parasite carries an antibody halo that reaches into that same ring, so subtracting it suppresses exactly the objects the threshold must keep above it. Default 'none'.
- **`min_total_intensity`** *(optional)* — (float or None) - Minimum mean intensity in the post-permeabilisation channel for an object to count as a parasite at all. That antibody stains every parasite, so an object dark in it is debris inside the pathogen mask rather than a dim parasite, and it would otherwise contribute a background-level outside signal and be scored invaded. None applies no filter. Default None.

#### Thresholding

- **`outside_threshold_method`** *(optional)* — (str) - How the outside-stain cut is derived from each field's own objects when no fixed value and no control wells are given: 'otsu', 'triangle', 'li', 'yen' or 'mean'. All of them find a split; none of them can tell you a split exists, which is what the bimodality check is for. 'triangle' suits a heavily skewed distribution with a small stained minority, 'otsu' a more balanced one. Default 'otsu'.
- **`outside_threshold`** *(optional)* — (float or None) - Fixed cut on the outside-stain statistic, applied to every field and overriding both the automatic method and the control wells. Set it only when you have calibrated it yourself: raising it above the true cut moves attached parasites into the invaded class and inflates invasion efficiency, and nothing moves them back. Control wells, if given, stay on as the reference the QC judges the fixed value against. None derives the cut per field. Default None.
- **`threshold_agreement_tolerance`** *(optional)* — (float) - Relative distance a threshold may sit from its reference before the field and well are flagged; the reference is the control-derived cut when controls exist, otherwise the field's own automatic cut. 0.5 means a factor of two. Lower it to catch smaller drifts between a fixed threshold and what the data would have chosen. Default 0.5.
- **`threshold_sensitivity`** *(optional)* — (float) - Fractional amount the threshold is moved up and down to produce the invasion_efficiency_low_threshold and _high_threshold bracket, which shows how much of a well's answer is the threshold rather than the biology. Widening it widens the bracket and makes the inflation flag more eager. Default 0.25.
- **`bimodality_cutoff`** *(optional)* — (float) - Value of the bimodality coefficient above which a distribution counts as two populations rather than one, below which the field and well are flagged and their efficiency should not be quoted. A perfect two-population mixture scores 1.0 at any mixing ratio and a single normal population about 0.33, so 5/9 sits between them. Raise it to demand cleaner separation before a number is reported unflagged. Default 0.5555555555555556.
- **`extracellular_class`** *(optional)* — (str) - How parasites overlapping no host cell are scored. 'attached' calls them attached whatever the stain says, since something outside every cell cannot have invaded one; 'classify' leaves the decision to the stain, which is what you want when the cell mask is the unreliable part; 'exclude' drops them before anything is counted. The count is reported as n_no_host_cell either way, so the choice stays visible. Default 'attached'.

#### Controls & Minimum Counts

- **`control_wells`** *(conditionally required)* — (list or None) - Wells whose parasites carry no pre-permeabilisation stain, giving the honest negative distribution the cut should sit above -- better evidence than any automatic method. Name a column ('c12'), a row ('r1'), a well ('r1_c12') or a full plate key. These wells are dropped from every efficiency, since a staining control is not an experimental condition. None runs the automatic per-field method instead. NOTE the screen regression reads this same key for a different job, where it must be a list matching filter_value. Default None.
- **`control_quantile`** *(optional)* — (float) - Quantile of the control wells' outside-stain distribution taken as the threshold. 0.99 means one in a hundred genuinely unstained parasites is misread as attached. Lowering it toward 0.95 buys safety against this assay's dangerous error - an outside parasite scored invaded - at the cost of a few false attached calls; raising it does the reverse. Default 0.99.
- **`min_control_objects`** *(optional)* — (int) - Objects a plate's control wells must contribute before their quantile is trusted as a threshold. Below it the plate falls back to the automatic per-field method and says so, rather than taking a 99th percentile from a handful of points. Default 10.
- **`min_objects_for_threshold`** *(optional)* — (int) - Objects a field must hold before a threshold is derived from it alone; below it the field borrows its well's threshold, then its plate's, and the level actually used is written into automatic_source. Raising it makes thresholds steadier and less local, which is the wrong trade when illumination varies across the field of view. Default 10.
- **`min_objects_for_bimodality`** *(optional)* — (int) - Objects required before the bimodality coefficient is computed at all; below it the coefficient is left NaN and the field or well is flagged. The statistic exceeds its cutoff on genuinely unimodal data about 45% of the time at ten objects and 15% at twenty, so computing it there would silence the check exactly where the classification is least trustworthy. Default 30, where that false-pass rate is 5%.
- **`min_parasites_per_well`** *(optional)* — (int) - Scored parasites below which a well's efficiency is flagged as too thin to quote. At fifty the 95% interval on a proportion near a half is still about fourteen percentage points wide, which is larger than most real effects. The number is never suppressed, only marked, and n_total travels beside it. Default 50.
- **`inflation_warn`** *(optional)* — (float) - Extra invasion efficiency, in proportion units, that raising the threshold by threshold_sensitivity may add to a well before the well is flagged. Only the upward move is watched, because lowering a threshold can only turn invaded back into attached and never invents a result. 0.05 flags a well whose efficiency would gain more than five percentage points. Default 0.05.

#### Object Filtering

- **`min_parasite_area`** *(optional)* — (int or float) - Smallest object area in pixels kept as a parasite. Smaller objects are debris, and their outside-stain statistic is noise over a handful of pixels that will land on whichever side of the threshold the noise happens to fall. Raise it when the pathogen mask is shattering. Default 0, which filters nothing.
- **`max_parasite_area`** *(optional)* — (float or None) - Largest object area in pixels kept as a parasite. Anything bigger is several parasites merged by the mask, whose rim statistic mixes them and whose single classification then stands for all of them. None keeps everything. Default None.

#### Condition Metadata

- **`cell_types`** *(optional)* — (list) - Names of the host cell lines in the experiment, e.g. ['HeLa']. Each name is written into the host_cells column and folded into the combined condition label used for grouping and plotting; the list is positionally paired with cell_plate_metadata, which says which wells hold each one. Default ['HeLa'].
- **`cell_plate_metadata`** *(optional)* — (list of lists) - Wells occupied by each entry of cell_types, one inner list per cell type in the same order, e.g. [['c2','c3'],['c4']]. Every identifier must start with 'c' (column) or 'r' (row); anything else is SILENTLY skipped and those wells get no host_cells label. An unlabelled well is not lost -- 'condition' joins whichever labels do exist -- so a typo here quietly changes what is being compared rather than raising. Default None.
- **`pathogen_types`** *(required)* — (list) - Names given to each pathogen condition on the plate, e.g. ['wt','ku80']. Element i is written into the pathogen column for every well listed in pathogen_plate_metadata[i] and folded into the combined condition label used for grouping and plotting. Must match pathogen_plate_metadata in length and order; None skips pathogen annotation. Default ['pathogen_1', 'pathogen_2'] for the dataset builders, ['pc'] for the control-based paths, None where types are not used.
- **`pathogen_plate_metadata`** *(required)* — (list of lists) - Well locations of each pathogen condition, one inner list per entry in pathogen_types. Every item must be a row or column ID string such as 'c1' or 'r3'; anything else is silently ignored and those wells stay unannotated. Ranges like 'c2-c11' are not expanded - list each row/column. Do not leave it None while pathogen_types is set: annotation is not skipped, every row is labelled with the first pathogen_types entry. Defaults: None in the plot-from-db settings, [['c1','c2','c3'],['c4','c5','c6']] for recruitment analysis.
- **`treatments`** *(optional)* — (list) - Names of the drug or treatment conditions in the experiment, e.g. ['dmso','lovastatin']. Each name is written into the treatment column and folded into the combined condition label used for grouping and plotting; positionally paired with treatment_plate_metadata (or treatment_loc), which lists the wells for each. Default ['cm','lovastatin'].
- **`treatment_plate_metadata`** *(optional)* — (list of lists) - Wells that received each entry of treatments, one inner list per treatment in the same order, e.g. [['r1','r2','r3'],['r4','r5','r6']]. Entries must start with 'r' (row) or 'c' (column); anything else is IGNORED and those wells get no treatment label rather than an error. Wells you do not list are still kept -- 'condition' joins whatever cell/pathogen/treatment labels exist, so an unlisted well simply carries fewer. Default None.
- **`group_column`** *(optional)* — (str) - Column whose values become the experimental conditions compared against each other; 'condition' is the combined host-cell / pathogen / treatment label built from the plate-metadata maps. Point it at 'pathogen' or 'treatment' to compare on one factor alone. Rows with no value here are dropped before anything is counted. Default 'condition'.
- **`level`** *(optional)* — (str) - Select the regression fit level or the proportion-summary unit. Regression: 'both' runs separate gRNA and gene fits, writes results_grna.csv and results_gene.csv, and corrects each fit independently with multiple_testing_method. This avoids a collinear combined design because a gene fraction is the sum of its guide fractions. 'grna' or 'gene' runs one fit. Disabled for mixed models, which nest guides within genes. Proportion plots: 'object' pools objects, 'well' averages by well, and 'plate' averages by plate. Default 'both' for regression and 'object' for proportions.
- **`change_plate`** *(optional)* — (bool) - Relabel each source directory as plate1, plate2, ... instead of trusting the plate ID stored in its database. Use it when several plates were written under the same name, which would otherwise let two plates' fields pool into one threshold and one well. Default False.

#### Assay Output

- **`cmap`** *(optional)* — (str) - Matplotlib colormap applied to single-channel image previews and to plate heatmaps. Perceptually uniform maps ('viridis', 'inferno', 'magma') keep intensity differences honest; 'gray' matches how the raw microscope data looks. Any registered matplotlib name works, with an '_r' suffix to reverse it. Default 'inferno' for image plots, 'viridis' for plate heatmaps.
- **`qc_plot_max_panels`** *(optional)* — (int) - Largest number of wells drawn in the threshold-diagnostic figure, taken in sorted well order. It exists so a 384-well plate does not produce a 384-panel figure; the CSVs always carry every well regardless. Default 12.
- **`seed_wells_from_cells`** *(optional)* — (bool) - Read the cell table as well, so a well holding host cells but no parasites appears in the results with a zero denominator instead of vanishing from the plate entirely. Switch it off only when the database has no cell table. Default True.
- **`save`** *(optional)* — (bool or list of bool) - Whether to save masks to disk. Can be a list of three booleans for [cell, nucleus, pathogen] independently. Default varies by module -- False for most, True for the sequencing and regression paths.

#### Runtime & Reliability

- **`verbose`** *(optional)* — (bool) - Print extra run detail instead of the minimal log: the resolved settings table, the channel and model choices per object type, per-table row counts, and how many objects survive each filter. It only adds console output, so turn it on when object counts come out unexpected and you need to see which stage removed them. The default differs per pipeline -- True for mask, UMAP, screen analysis, barcode mapping and Cellpose training; False for measure, the plotting helpers and regression.

## 4. Run

After editing the settings cell, run the function cell immediately below it. Long operations report progress through spaCR's logging system; set `SPACR_LOG_LEVEL=DEBUG` before starting Jupyter for more detail.

In [ ]:
settings = {
    # Assay Inputs
    # Required settings
    'src': 'path',
    # Optional settings
    'parasite_table': 'pathogen',
    'compartment': 'pathogen',

    # Channels & Intensity
    # Required settings
    'outside_channel': 1,
    'total_channel': 0,
    # Optional settings
    'intensity_statistic': 'auto',
    'background_correction': 'none',
    'min_total_intensity': None,

    # Thresholding
    # Optional settings
    'outside_threshold_method': 'otsu',
    'outside_threshold': None,
    'threshold_agreement_tolerance': 0.5,
    'threshold_sensitivity': 0.25,
    'bimodality_cutoff': 0.5555555555555556,
    'extracellular_class': 'attached',

    # Controls & Minimum Counts
    # Conditionally required settings
    'control_wells': None,
    # Optional settings
    'control_quantile': 0.99,
    'min_control_objects': 10,
    'min_objects_for_threshold': 10,
    'min_objects_for_bimodality': 30,
    'min_parasites_per_well': 50,
    'inflation_warn': 0.05,

    # Object Filtering
    # Optional settings
    'min_parasite_area': 0,
    'max_parasite_area': None,

    # Condition Metadata
    # Required settings
    'pathogen_types': ['pc'],
    'pathogen_plate_metadata': [['c1'], ['c2']],
    # Optional settings
    'cell_types': ['Hela'],
    'cell_plate_metadata': None,
    'treatments': None,
    'treatment_plate_metadata': None,
    'group_column': 'condition',
    'level': 'object',
    'change_plate': False,

    # Assay Output
    # Optional settings
    'cmap': 'viridis',
    'qc_plot_max_panels': 12,
    'seed_wells_from_cells': True,
    'save': True,

    # Runtime & Reliability
    # Optional settings
    'verbose': False,
}

In [ ]:
analyze_invasion(settings)

## Outputs and next steps

Per-object invasion labels and per-well or per-condition invasion estimates.

Output directories remain associated with the source dataset, which preserves plate-level provenance across subsequent spaCR workflows.

### Related documentation

- [Graphical workflow tutorials](https://einarolafsson.github.io/spacr/tutorials/)
- [Python API reference](https://einarolafsson.github.io/spacr/python_api.html)